# 01 - Scrape Wuzzuf Jobs (Raw)

Stage 1 of the pipeline.

1. Collects job-detail URLs from the Wuzzuf Saudi Arabia search results, skipping anything already in the raw history (incremental).
2. Downloads each new job page and pulls the raw `job` and `company` entities out of Wuzzuf's embedded `initialStoreState` JSON blob.
3. Writes the new jobs to **two** places:
   - `RAW_HISTORY` — every job ever scraped, deduplicated by `job_url`. This is the full audit trail.
   - `PENDING_RAW_QUEUE` — only jobs the cleaning notebook hasn't consumed yet. Each scrape run **appends** to this queue rather than overwriting it, so if you scrape more often than you clean, nothing is lost — batches just pile up until `02_clean_wuzzuf.ipynb` runs and clears it.

No localization, formatting, or schema normalization happens here on purpose — that's the cleaning notebook's job.

In [ ]:
import json
import logging
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

## Settings

In [ ]:
# SEARCH SETTINGS
SEARCH_BASE_URL = (
    "https://wuzzuf.net/saudi/search/jobs"
    "?filters%5Bcountry%5D%5B0%5D=Saudi%20Arabia"
)

SEARCH_PARAMS = {
    "q": "",
    "a": "spbg",
}

# OUTPUT
RAW_HISTORY = "../data/raw/wuzzuf_jobs_raw.json"              # full raw history (audit)
PENDING_RAW_QUEUE = "../data/raw/wuzzuf_pending_raw_queue.json"  # unconsumed batch for notebook 2
FAILED_URLS_OUTPUT = "../data/raw/wuzzuf_failed_urls.txt"

# REQUEST / RETRY SETTINGS
REQUEST_TIMEOUT = 20
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 5

LISTING_PAGE_DELAY = 3.0
DETAIL_PAGE_DELAY = 0.5

DEFAULT_WORKERS = 4

# RUN LIMITS (set to None to remove the limit)
MAX_PAGES = None
MAX_JOBS = None
WORKERS = DEFAULT_WORKERS

# INCREMENTAL MODE
# True  -> skip jobs already in RAW_HISTORY, stop paginating once a search
#          page has no unseen jobs left (fast repeat runs).
# False -> ignore history and re-scrape everything from scratch.
INCREMENTAL = True

In [ ]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,ar;q=0.5",
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8"
    ),
}

STORE_STATE_PATTERN = re.compile(
    r"Wuzzuf\.initialStoreState\s*=\s*(\{.*?\})\s*;\s*\n",
    re.DOTALL,
)

STORE_STATE_FALLBACK_PATTERN = re.compile(
    r"Wuzzuf\.initialStoreState\s*=\s*(\{.*\})",
    re.DOTALL,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("wuzzuf.scrape")

## HTTP helpers

In [ ]:
def make_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def fetch_html(session, url, params=None):
    """GET a URL with retries and backoff. Returns HTML text or None."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, params=params, timeout=REQUEST_TIMEOUT)

            if response.status_code != 200:
                log.warning(
                    "Non-200 (%s) for %s (attempt %d/%d)",
                    response.status_code, response.url, attempt, MAX_RETRIES,
                )
                if response.status_code in (403, 429):
                    time.sleep(RETRY_BACKOFF_SECONDS * attempt)
                    continue
                return None

            response.encoding = response.apparent_encoding or "utf-8"
            return response.text

        except requests.exceptions.RequestException as e:
            log.warning(
                "Request failed (%s) attempt %d/%d: %s",
                url, attempt, MAX_RETRIES, e,
            )
            time.sleep(RETRY_BACKOFF_SECONDS * attempt)

    return None

## Stage 1 - collect job URLs from search results

`known_urls` is every `job_url` already in `RAW_HISTORY`. A URL in `known_urls` is skipped, and once a whole search page turns out to be nothing but already-known URLs, pagination stops — Wuzzuf lists newest jobs first, so everything after that point should already be scraped too.

In [ ]:
def collect_job_urls(session, max_pages=None, known_urls=None):
    """Paginate through Wuzzuf search results and collect unique, unseen job URLs."""
    known_urls = known_urls or set()

    job_urls = []
    seen = set()
    page = 0

    while True:
        if max_pages is not None and page >= max_pages:
            log.info("Reached max_pages=%d. Stopping URL collection.", max_pages)
            break

        params = dict(SEARCH_PARAMS)
        if page > 0:
            params["start"] = page

        log.info("Scraping search page %d...", page + 1)

        html = fetch_html(session, SEARCH_BASE_URL, params=params)
        if html is None:
            log.error("Search page %d failed after retries. Stopping.", page + 1)
            break

        soup = BeautifulSoup(html, "html.parser")
        page_urls = []
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "/jobs/p/" in href:
                url = urljoin("https://wuzzuf.net", href)
                url = url.replace("https://wuzzuf.net/ar/", "https://wuzzuf.net/")
                page_urls.append(url)

        new_this_run = [u for u in dict.fromkeys(page_urls) if u not in seen]
        unseen_overall = [u for u in new_this_run if u not in known_urls]

        log.info(
            "  found %d links (%d new this run, %d not previously scraped)",
            len(page_urls), len(new_this_run), len(unseen_overall),
        )

        if not new_this_run:
            log.info("No new links on this page at all. Stopping pagination.")
            break

        seen.update(new_this_run)

        if known_urls and not unseen_overall:
            log.info(
                "Every link on this page was already scraped in a previous "
                "run. Stopping pagination (incremental)."
            )
            break

        job_urls.extend(unseen_overall)
        log.info("  running total: %d new unique job URLs", len(job_urls))

        page += 1
        time.sleep(LISTING_PAGE_DELAY)

    return job_urls

## Store-state extraction (Wuzzuf embeds job data as inline JSON)

In [ ]:
def extract_store_state(html):
    """Extract and parse Wuzzuf.initialStoreState JSON from the page HTML."""
    match = STORE_STATE_PATTERN.search(html)
    if not match:
        match = STORE_STATE_FALLBACK_PATTERN.search(html)
    if not match:
        raise ValueError("Could not find 'Wuzzuf.initialStoreState' in the page HTML.")

    raw_json = match.group(1)
    try:
        return json.loads(raw_json)
    except json.JSONDecodeError:
        return json.loads(_trim_to_balanced_json(raw_json))


def _trim_to_balanced_json(text):
    start = text.find("{")
    if start == -1:
        raise ValueError("No opening brace found in store-state blob.")

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        char = text[i]

        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    raise ValueError("Could not find a balanced closing brace for the JSON blob.")

In [ ]:
import hashlib


def generate_job_id(job_url):
    """Generate a stable job_id from the Wuzzuf job URL slug."""
    slug_match = re.search(r"/jobs/p/([^/?]+)", job_url)
    if slug_match:
        slug = slug_match.group(1)
        return hashlib.sha1(slug.encode("utf-8")).hexdigest()[:12]
    return hashlib.sha1(job_url.encode("utf-8")).hexdigest()[:12]


def find_job_id_by_slug(store, url):
    """Match the correct job entity in the store state using the URL slug."""
    slug_match = re.search(r"/p/([^/?]+)", url)
    target_slug = slug_match.group(1) if slug_match else None

    job_collection = store.get("entities", {}).get("job", {}).get("collection", {})

    if target_slug:
        for job_id, job_entity in job_collection.items():
            attrs = job_entity.get("attributes", {})
            if attrs.get("slug") == target_slug:
                return job_id, job_entity

    similar_jobs = store.get("jobPage", {}).get("similarJobs", {})
    if similar_jobs:
        main_job_id = next(iter(similar_jobs))
        job_entity = job_collection.get(main_job_id)
        if job_entity:
            return main_job_id, job_entity

    if job_collection:
        job_id = next(iter(job_collection))
        return job_id, job_collection[job_id]

    return None, None


def get_company(store, job_entity):
    company_ref = job_entity.get("relationships", {}).get("company", {}).get("data", {})
    if not company_ref:
        return None
    company_id = company_ref.get("id")
    return store.get("entities", {}).get("company", {}).get("collection", {}).get(company_id)

## Stage 2 - scrape one job page and keep the **raw** attributes

This deliberately does *not* localize / format anything. It just pulls `attributes` off the job and company entities so the cleaning notebook has the full raw payload to work with.

In [ ]:
def scrape_job_raw(session, url):
    """Fetch one job page and return its raw job/company attributes."""
    try:
        html = fetch_html(session, url)
        if html is None:
            log.error("Failed to download job page: %s", url)
            return None

        store_state = extract_store_state(html)

        wuzzuf_job_id, job_entity = find_job_id_by_slug(store_state, url)
        if not job_entity:
            log.error("Could not locate job entity in store state: %s", url)
            return None

        company_entity = get_company(store_state, job_entity)

        return {
            "job_id": generate_job_id(url),
            "job_url": url,
            "wuzzuf_job_id": wuzzuf_job_id,
            "job_attributes": job_entity.get("attributes", {}),
            "company_attributes": (company_entity or {}).get("attributes", {}),
            "scraped_at": datetime.now(timezone.utc).replace(tzinfo=None).strftime("%Y-%m-%d %H:%M:%S"),
        }

    except Exception as e:
        log.error("Error parsing %s: %s: %s", url, type(e).__name__, e)
        return None

## Queue / history helpers

`dedupe_keep_last` merges two lists of records on `job_url`, keeping the most recently-scraped version of any duplicate. It's what makes every append idempotent: re-running a stage on data it has already seen never creates duplicate entries in a history file.

In [ ]:
def load_json(path):
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(records, path):
    dirname = os.path.dirname(path)
    if dirname:
        os.makedirs(dirname, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def dedupe_keep_last(records, key="job_url"):
    """Merge records, keeping the most recent version of each key."""
    deduped = {}
    for r in records:
        k = r.get(key)
        if k is None:
            continue
        deduped[k] = r
    return list(deduped.values())


def save_urls(urls, filename):
    dirname = os.path.dirname(filename)
    if dirname:
        os.makedirs(dirname, exist_ok=True)
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(urls))

## Run the scrape

`run_scraper`:
1. Loads `RAW_HISTORY` and uses its `job_url`s as `known_urls`, so `collect_job_urls` only returns genuinely new postings.
2. Scrapes just those.
3. Merges the new records into `RAW_HISTORY` (dedup by `job_url`) **and** appends them onto `PENDING_RAW_QUEUE` (also dedup by `job_url`, so a second run before the queue is consumed just adds to it rather than duplicating anything).

In [ ]:
def run_scraper(max_pages=None, max_jobs=None, workers=DEFAULT_WORKERS, incremental=True):
    session = make_session()

    raw_history = load_json(RAW_HISTORY) if incremental else []
    known_urls = {r["job_url"] for r in raw_history if r.get("job_url")}

    if incremental:
        log.info("Incremental mode: %d jobs already in raw history will be skipped.", len(known_urls))
    else:
        log.info("Incremental mode off: re-scraping from scratch.")

    log.info("=" * 70)
    log.info("STAGE 1: Collecting job URLs from search results")
    log.info("=" * 70)

    job_urls = collect_job_urls(session, max_pages=max_pages, known_urls=known_urls)

    if max_jobs is not None:
        job_urls = job_urls[:max_jobs]
        log.info("Limiting to first %d new jobs for detail scraping.", len(job_urls))

    if not job_urls:
        log.info("No new job URLs found. Nothing to scrape.")
        return raw_history, []

    log.info("=" * 70)
    log.info("STAGE 2: Scraping raw details for %d new jobs (workers=%d)", len(job_urls), workers)
    log.info("=" * 70)

    new_records = []
    failed_urls = []

    def worker(url):
        thread_session = make_session()
        result = scrape_job_raw(thread_session, url)
        time.sleep(DETAIL_PAGE_DELAY)
        return url, result

    def flush_progress():
        """Merge what's been scraped so far into the history + pending queue."""
        merged_history = dedupe_keep_last(raw_history + new_records, key="job_url")
        save_json(merged_history, RAW_HISTORY)

        pending = load_json(PENDING_RAW_QUEUE)
        pending = dedupe_keep_last(pending + new_records, key="job_url")
        save_json(pending, PENDING_RAW_QUEUE)

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(worker, url): url for url in job_urls}
        done = 0

        for future in as_completed(futures):
            url, result = future.result()
            done += 1

            if result:
                new_records.append(result)
                title = (result.get("job_attributes") or {}).get("title") or url
                log.info("[%d/%d] OK - %s", done, len(job_urls), title)
            else:
                failed_urls.append(url)
                log.info("[%d/%d] FAIL - %s", done, len(job_urls), url)

            if done % 10 == 0 or done == len(job_urls):
                flush_progress()

    flush_progress()
    final_history = load_json(RAW_HISTORY)

    log.info("=" * 70)
    log.info("DONE")
    log.info("=" * 70)
    log.info("New jobs scraped this run: %d / %d attempted", len(new_records), len(job_urls))
    log.info("Failed this run: %d jobs", len(failed_urls))
    log.info("Raw history total: %d | Pending raw queue: %d", len(final_history), len(load_json(PENDING_RAW_QUEUE)))

    if failed_urls:
        save_urls(failed_urls, FAILED_URLS_OUTPUT)
        log.info("Failed URLs saved to: %s", FAILED_URLS_OUTPUT)

    return final_history, failed_urls

In [ ]:
raw_history, failed_urls = run_scraper(
    max_pages=MAX_PAGES,
    max_jobs=MAX_JOBS,
    workers=WORKERS,
    incremental=INCREMENTAL,
)

len(raw_history), len(failed_urls)